# **PART 2. DATA PREPROCESSING**

## **1. Overview of preprocessing and data exploration**
- Handling missing data
- Noise handling
- Standardize data
-  Give first glimpses of data

## **2. Read the original obtained data file**


### Import libraries

In [1]:
#import library
import requests
import numpy as np
import pandas as pd

In [2]:
#Read data
data_origin = pd.read_csv("data_origin.csv")
print(data_origin)

                                           Product Name  \
0     Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...   
1     Dung dịch vệ sinh vùng kín Bimunica 250ml dành...   
2     Kem giảm thâm vùng nách, mông, bikini Neothera...   
3     Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...   
4     Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...   
...                                                 ...   
1994  Viên uống Tiền Đình Khang KGA dùng cho rối loạ...   
1995  Viên nang mềm Xương Khớp PV Forte hỗ trợ khớp,...   
1996  Gói cốm Futizon PV Pharma hỗ trợ bù nước, điện...   
1997  Sữa Pregestimil Lipil Mead Johnson bổ sung din...   
1998  Viên sủi Haas Vital Pezsgés Multi-vitamin hươn...   

                      Category        Dosage Form     Price     Trademark  \
0              Chăm sóc cơ thể                Gel  105000.0       DECUMAR   
1              Chăm sóc cơ thể                NaN  230000.0           NaN   
2              Chăm sóc cơ thể                NaN  139000.0 

- Obtain 1999 rows and 8 columns

## Does the raw data have duplicate rows?

In [3]:
# Check if data have duplicate rows
num_duplicated_rows = data_origin.duplicated().sum()
if num_duplicated_rows == 0:
    print(f"Your raw data have no duplicated line.!")
else:
    if num_duplicated_rows > 1:
        ext = "lines"
    else:
        ext = "line"
    print(f"Your raw data have {num_duplicated_rows} duplicated " + ext + ". Please de-deduplicate your raw data.!")

Your raw data have no duplicated line.!


- After checking, there are **no duplicate columns**.

## **3. What data type does each column currently have? Are there any columns whose data types are not suitable for further processing?**

In [4]:
#Type of each column
dtypes = data_origin.dtypes

- Product Name     object
- Category         object
- Dosage Form      object
- Price           float64
- Trademark        object
- Brand Origin     object
- Country          object
- Rating          float64


- dtype: object

## **4. Check the percentage of missing data in the columns**

In [5]:
#Percentage of missing data
missing_percentage = data_origin.isnull().mean() * 100
print("Missing ratio")
print(missing_percentage)

Missing ratio
Product Name     0.000000
Category         0.000000
Dosage Form     24.612306
Price           25.112556
Trademark        8.354177
Brand Origin     5.252626
Country          0.500250
Rating          38.969485
dtype: float64


### **Missing ratio**
|Attribute           |Percentage of missing values (%)|
|--------------------|:-----------:|
|**Product Name**    |0.000000|
|**Category**         |0.000000|
|**Dosage Form**     |24.612306|
|**Price**           |25.112556|
|**Trademark**        |8.354177|
|**Brand Origin**     |5.252626|
|**Country**          |0.500250|
|**Rating**          |38.969485|

- After identifying the basic statistical numbers that describe data, we further need to determine the features that have a large number of missing values. Such features are not useful for the analysis stage and must be removed from the dataset.

- Depending on goals, the threshold for "large" can be defined. Usually, if the percentage of missing values is greater than 75%, the column is dropped from the dataframe and an updated dataframe is returned.

In [6]:
def drop_missing_features(df: pd.DataFrame, missing_percentage: pd.Series, threshold: float = 75.0) -> pd.DataFrame:
    # Find columns with missing data percentage greater than the threshold
    cols_to_drop = missing_percentage[missing_percentage > threshold].index
    # Drop those columns from the DataFrame
    return df.drop(columns=cols_to_drop)

# Apply the function to the dataframe `data_origin` using the `missing_percentage` series
raw_df = drop_missing_features(data_origin, missing_percentage)

# Display the first few rows of the resulting dataframe
raw_df.head()


,Product Name,Category,Dosage Form,Price,Trademark,Brand Origin,Country,Rating
0,"Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...",Chăm sóc cơ thể,Gel,105000.0,DECUMAR,Việt Nam,Việt Nam,5.0
1,Dung dịch vệ sinh vùng kín Bimunica 250ml dành...,Chăm sóc cơ thể,NaN,230000.0,NaN,Hoa Kỳ,Liên Bang Nga,5.0
2,"Kem giảm thâm vùng nách, mông, bikini Neothera...",Chăm sóc cơ thể,NaN,139000.0,La Beauty,Việt Nam,Việt Nam,5.0
3,Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...,Chăm sóc cơ thể,NaN,390000.0,SVR,Pháp,Pháp,NaN
4,Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...,"Lăn khử mùi, xịt khử mùi",Dạng bọt,96000.0,NaN,Việt Nam,Việt Nam,5.0


##### After determining the percentage of missing data in the columns, we will now **divide them into two categories: numeric data type and non-numeric data type** for processing.

## **5. For each column with numeric data type, how are the values distributed?**

For columns with numeric data types, we will calculate:
- Percentage (from 0 to 100) of missing values
- The min
- The lower quartile
- The median
- The upper quartile
- The max

Column with **numeric data**: **Price** and **Rating**

In [7]:
def missing_ratio(column):
    return column.isna().mean() * 100

def lower_quartile(column):
    return column.quantile(0.25)

def median(column):
    return column.median()

def upper_quartile(column):
    return column.quantile(0.75)

# Select numerical columns (float64, int64) from the DataFrame
numeric_cols = data_origin.select_dtypes(include=['float64', 'int64']).columns

# Dictionary to store the statistics
statistics = {
    "missing_ratio": [],
    "min": [],
    "lower_quartile": [],
    "median": [],
    "upper_quartile": [],
    "max": []
}

# Calculate statistics for each numerical column
for col in numeric_cols:
    missing_ratio_val = missing_ratio(data_origin[col])
    min_val = data_origin[col].min()
    lower_quartile_val = lower_quartile(data_origin[col])
    median_val = median(data_origin[col])
    upper_quartile_val = upper_quartile(data_origin[col])
    max_val = data_origin[col].max()
    
    statistics["missing_ratio"].append(missing_ratio_val)
    statistics["min"].append(min_val)
    statistics["lower_quartile"].append(lower_quartile_val)
    statistics["median"].append(median_val)
    statistics["upper_quartile"].append(upper_quartile_val)
    statistics["max"].append(max_val)

# Create a DataFrame from the statistics dictionary
num_col_info_df = pd.DataFrame(statistics, index=numeric_cols).T.round(1)
num_col_info_df


,Price,Rating
missing_ratio,25.1,39.0
min,30.0,1.0
lower_quartile,120000.0,5.0
median,230000.0,5.0
upper_quartile,392000.0,5.0
max,6675000.0,5.0


- **Price column**: For the missing rows, fill them in by calculating the average of the entire column and then filling in the missing rows. 
- **Rating column**: For the products that are not rated, change 'null' to 'unknown'. This helps to divide into two parts: products that are rated and products that are not rated.

In [8]:
# Replace null = median value
raw_df['Price'] = raw_df['Price'].fillna(raw_df['Price'].median())

# Replace null = "unknown"
raw_df['Rating'] = raw_df['Rating'].fillna('unknown')

## **6. For each column with a non-numeric data type, how are the values distributed?**

In [9]:
non_numeric_cols = raw_df.select_dtypes(exclude=['float64', 'int64']).columns
cat_statistics = {
    "missing_ratio": [],
    "num_values": [],  #Numbers of unique values
}
for col in non_numeric_cols:
    
    missing_ratio = raw_df[col].isna().mean() * 100

    num_values = raw_df[col].nunique()

    cat_statistics["missing_ratio"].append(round(missing_ratio, 1))
    cat_statistics["num_values"].append(num_values)

    
cat_col_info_df = pd.DataFrame(cat_statistics, index=non_numeric_cols).T
print (cat_col_info_df)

               Product Name  Category  Dosage Form  Trademark  Brand Origin  \
missing_ratio           0.0       0.0         24.6        8.4           5.3   
num_values           1999.0      94.0         46.0      484.0          39.0   

               Country  Rating  
missing_ratio      0.5     0.0  
num_values        45.0    17.0  


### For non-numeric columns: Find the most frequently occurring word in the column and replace the missing values with it.

In [10]:
def filling_missing_value(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=['category', 'object']).columns:
        df[col] = df[col].fillna(df[col].mode()[0])
    return df
raw_df = filling_missing_value(df=raw_df)

### Check again after filled null values

In [11]:
missing_percentage_aftercheck = raw_df.isnull().mean() * 100
print("Missing ratio")
print(missing_percentage_aftercheck)

Missing ratio
Product Name    0.0
Category        0.0
Dosage Form     0.0
Price           0.0
Trademark       0.0
Brand Origin    0.0
Country         0.0
Rating          0.0
dtype: float64


- After completing the steps to fill in the missing data, the percentage of missing values in the columns is now 0%. This ensures that the original number of rows and columns is maintained.

### Realizing that the existing attributes were insufficient for Making questions and Visualizing, our team decided to increase the number of attributes.
- Added the "General_function" column: Instead of having up to 94 different categories, we will group them into functionally similar groups.

In [12]:
category_to_group = {
    'Chăm sóc cơ thể': 'Chăm sóc cơ thể',
    'Lăn khử mùi, xịt khử mùi': 'Chăm sóc cơ thể',
    'Sữa rửa mặt (Kem, gel, sữa)': 'Chăm sóc da mặt',
    'Nước tẩy trang, dầu tẩy trang': 'Chăm sóc da mặt',
    'Tẩy tế bào chết': 'Hỗ trợ làm đẹp',
    'Trị sẹo, mờ vết thâm': 'Hỗ trợ làm đẹp',
    'Dầu gội dầu xả': 'Chăm sóc tóc - da đầu',
    'Sữa tắm, xà bông': 'Chăm sóc cơ thể',
    'Sữa dưỡng thể, kem dưỡng thể': 'Chăm sóc cơ thể',
    'Dưỡng da bị khô, thiếu ẩm': 'Chăm sóc cơ thể',
    'Chống nắng toàn thân': 'Chăm sóc cơ thể',
    'Chăm sóc tóc - da đầu': 'Chăm sóc tóc - da đầu',
    'Chăm sóc da mặt': 'Chăm sóc da mặt',
    'Dưỡng tóc, ủ tóc': 'Chăm sóc tóc - da đầu',
    'Kem trị mụn, gel trị mụn': 'Chăm sóc da mặt',
    'Dầu gội trị nấm': 'Chăm sóc tóc - da đầu',
    'Giải pháp làn da': 'Giải pháp làn da',
    'Kem trị nám, tàn nhang, đốm nâu': 'Hỗ trợ làm đẹp',
    'Son môi': 'Mỹ phẩm trang điểm',
    'Trị nứt da': 'Hỗ trợ làm đẹp',
    'Tinh dầu': 'Sản phẩm từ thiên nhiên',
    'Mặt nạ': 'Chăm sóc da mặt',
    'Dưỡng da mặt': 'Chăm sóc da mặt',
    'Kem dưỡng da tay, chân': 'Chăm sóc cơ thể',
    'Massage': 'Chăm sóc cơ thể',
    'Da sạm, xỉn màu': 'Hỗ trợ làm đẹp',
    'Mỹ phẩm trang điểm': 'Mỹ phẩm trang điểm',
    'Kem chống nắng da mặt': 'Chăm sóc da mặt',
    'Toner (nước hoa hồng) / Lotion': 'Chăm sóc da mặt',
    'Dưỡng da mắt': 'Chăm sóc da vùng mắt',
    'Xoá nếp nhăn vùng mắt': 'Chăm sóc da vùng mắt',
    'Viêm da cơ địa': 'Giải pháp làn da',
    'Dầu dừa': 'Sản phẩm từ thiên nhiên',
    'Serum, Essence hoặc Ampoule': 'Chăm sóc da mặt',
    'Chăm sóc ngực': 'Chăm sóc cơ thể',
    'Da bị kích ứng': 'Giải pháp làn da',
    'Xịt khoáng': 'Giải pháp làn da',
    'Dược mỹ phẩm': 'Chăm sóc da mặt',
    'Tái tạo, chống lão hóa da': 'Hỗ trợ làm đẹp',
    'Trị quầng thâm, bọng mắt': 'Hỗ trợ làm đẹp',
    'Trang điểm mặt': 'Mỹ phẩm trang điểm',
    'Đặc trị cho tóc': 'Chăm sóc tóc - da đầu',
    'Thận, tiền liệt tuyến': 'Cải thiện tăng cường chức năng',
    'Tăng sức đề kháng, miễn dịch': 'Cải thiện tăng cường chức năng',
    'Bổ sung Canxi & Vitamin D': 'Vitamin và Khoáng chất',
    'Vitamin & Khoáng chất': 'Vitamin và Khoáng chất',
    'Cải thiện tăng cường chức năng': 'Cải thiện tăng cường chức năng',
    'Bổ não - cải thiện trí nhớ': 'Cải thiện tăng cường chức năng',
    'Bổ mắt, bảo vệ mắt': 'Cải thiện tăng cường chức năng',
    'Tuần hoàn máu': 'Cải thiện tăng cường chức năng',
    'Cơ xương khớp': 'Cải thiện tăng cường chức năng',
    'Hô hấp, ho, xoang': 'Cải thiện tăng cường chức năng',
    'Hỗ trợ làm đẹp': 'Hỗ trợ làm đẹp',
    'Chức năng gan': 'Cải thiện tăng cường chức năng',
    'Thần kinh não': 'Thần kinh não',
    'Giải rượu, cai rượu': 'Thần kinh não',
    'Vi sinh - Probiotic': 'Hỗ trợ tiêu hóa',
    'Dạ dày, tá tràng': 'Cải thiện tăng cường chức năng',
    'Hỗ trợ trao đổi chất': 'Cải thiện tăng cường chức năng',
    'Bổ sung Sắt & Axit Folic': 'Vitamin và Khoáng chất',
    'Táo bón': 'Hỗ trợ tiêu hóa',
    'Vitamin tổng hợp': 'Vitamin và Khoáng chất',
    'Cân bằng nội tiết tố': 'Sinh lý - Nội tiết tố',
    'Đại tràng': 'Hỗ trợ tiêu hóa',
    'Hỗ trợ điều trị trĩ': 'Hỗ trợ điều trị',
    'Sinh lý nữ': 'Sinh lý - Nội tiết tố',
    'Hỗ trợ giấc ngủ ngon': 'Thần kinh não',
    'Giảm Cholesterol': 'Cải thiện tăng cường chức năng',
    'Da': 'Cải thiện tăng cường chức năng',
    'Dầu cá, Omega 3, DHA': 'Vitamin và Khoáng chất',
    'Tóc': 'Cải thiện tăng cường chức năng',
    'Hỗ trợ điều trị': 'Hỗ trợ điều trị',
    'Hỗ trợ tiêu hóa': 'Hỗ trợ tiêu hóa',
    'Sức khoẻ tim mạch': 'Sức khỏe tim mạch',
    'Sinh lý nam': 'Sinh lý - Nội tiết tố',
    'Khó tiêu': 'Hỗ trợ tiêu hóa',
    'Hỗ trợ điều trị tiểu đường': 'Cải thiện tăng cường chức năng',
    'Vitamin E các loại': 'Vitamin và Khoáng chất',
    'Chống lão hóa': 'Hỗ trợ làm đẹp',
    'Bổ sung Kẽm & Magie': 'Vitamin và Khoáng chất',
    'Hỗ trợ điều trị gout': 'Cải thiện tăng cường chức năng',
    'Sức khoẻ tình dục': 'Sinh lý - Nội tiết tố',
    'Hỗ trợ giảm cân': 'Hỗ trợ điều trị',
    'Sinh lý - Nội tiết tố': 'Sinh lý - Nội tiết tố',
    'Vitamin C các loại': 'Vitamin và Khoáng chất',
    'Hỗ trợ mãn kinh': 'Sinh lý - Nội tiết tố',
    'Huyết áp': 'Cải thiện tăng cường chức năng',
    'Suy giãn tĩnh mạch': 'Cải thiện tăng cường chức năng',
    'Sữa': 'Dinh dưỡng',
    'Dinh dưỡng': 'Dinh dưỡng',
    'Hoạt huyết': 'Cải thiện tăng cường chức năng',
    'Hỗ trợ điều trị ung thư': 'Hỗ trợ điều trị',
    'Kiểm soát căng thẳng': 'Thần kinh não',
    'Dinh dưỡng trẻ em': 'Dinh dưỡng'
}

raw_df['General_function'] = raw_df['Category'].map(category_to_group)
raw_df['General_function'] = raw_df['General_function'].fillna('Others')

In [13]:
#Check
print (raw_df)

                                           Product Name  \
0     Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...   
1     Dung dịch vệ sinh vùng kín Bimunica 250ml dành...   
2     Kem giảm thâm vùng nách, mông, bikini Neothera...   
3     Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...   
4     Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...   
...                                                 ...   
1994  Viên uống Tiền Đình Khang KGA dùng cho rối loạ...   
1995  Viên nang mềm Xương Khớp PV Forte hỗ trợ khớp,...   
1996  Gói cốm Futizon PV Pharma hỗ trợ bù nước, điện...   
1997  Sữa Pregestimil Lipil Mead Johnson bổ sung din...   
1998  Viên sủi Haas Vital Pezsgés Multi-vitamin hươn...   

                      Category        Dosage Form     Price     Trademark  \
0              Chăm sóc cơ thể                Gel  105000.0       DECUMAR   
1              Chăm sóc cơ thể           Viên nén  230000.0      Kingphar   
2              Chăm sóc cơ thể           Viên nén  139000.0 

- Now the data have 1999 rows and 10 columns.

### After checking everything, we found that in the "country" column, there are two different values, 'Hoa Kỳ' and 'USA', both referring to the same country. Change them to the value 'Mỹ'.

In [14]:
data_origin['Brand Origin'] = data_origin['Brand Origin'].replace({'Hoa Kỳ': 'Mỹ'})
data_origin['Country'] = data_origin['Country'].replace({'Hoa Kỳ': 'Mỹ', 'USA': 'Mỹ'})


## 7. **Save data to proccesed file**


In [15]:
raw_df.to_csv('data_processed.csv',index=False, encoding='utf-8-sig')